In [48]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126


In [49]:
!pip install kagglehub

import kagglehub

dataset_path = kagglehub.dataset_download("maysee/mushrooms-classification-common-genuss-images")

Using Colab cache for faster access to the 'mushrooms-classification-common-genuss-images' dataset.


In [50]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models, datasets

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
import time
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

Выведем распределение изображений по классам

In [51]:
from collections import Counter

def class_distribution(dataset_path):
    dataset_path_correct = os.path.join(dataset_path, "Mushrooms")
    dataset = datasets.ImageFolder(dataset_path_correct)

    labels = [label for _, label in dataset.samples]
    class_counts = Counter(labels)

    for i, class_name in enumerate(dataset.classes):
        count = class_counts[i]
        print(f"{class_name}: {count}")

    total_images = sum(class_counts.values())
    print(f"Всего изображений {total_images}")
    return dataset.classes

class_names = class_distribution(dataset_path)

Agaricus: 353
Amanita: 750
Boletus: 1073
Cortinarius: 836
Entoloma: 364
Hygrocybe: 316
Lactarius: 1563
Russula: 1148
Suillus: 311
Всего изображений 6714


обрабатаем изображения, если с ним что-то не то, то возвращаем черное фото

In [53]:
class MyDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]

        try:
            with Image.open(path) as img:
                image = img.convert('RGB')

            if self.transform:
                image = self.transform(image)

            return image, label

        except Exception as e:
            print(f"Error loading {path}: {e}")
            if self.transform:
                image = torch.zeros(3, 224, 224)
            else:
                image = Image.new('RGB', (224, 224))
            return image, label

поменяем данные под дальнейшее обучение

In [54]:
dataset_path_correct = os.path.join(dataset_path, "Mushrooms")
dataset = datasets.ImageFolder(dataset_path_correct)

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

from sklearn.model_selection import train_test_split

paths = [sample[0] for sample in dataset.samples]
labels = [sample[1] for sample in dataset.samples]

X_train, X_test, y_train, y_test = train_test_split(paths, labels, test_size=0.2, random_state=42, stratify=labels)
#stratify обеспечивает одинаковые пропорции классов, на тест оставляем 20%

train_dataset = MyDataset(X_train, y_train, transform=transform)
test_dataset = MyDataset(X_test, y_test, transform=transform)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

для обучения будем использовать ResNet

включает Batch Normalization, который нам нужен, так как распределение данных по классам неравномерное

In [56]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, 9))
model = model.to(device)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

def accuracy(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            total += y.size(0) #суммируем кол-во элементов в текущем батче
            correct += (predicted == y).sum().item() #суммируем правильные предсказания
    return 100 * correct / total

num_epochs = 8
best_acc = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (x, y) in enumerate(train_dataloader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = F.cross_entropy(outputs, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_dataloader) #ошибка в эпохе
    train_acc = accuracy(model, train_dataloader, device) #точночть на обучающей
    test_acc = accuracy(model, test_dataloader, device) #точность на тестовой

    scheduler.step()

    print(f'Epoch {epoch+1}/{num_epochs}:')
    print(f'Loss: {epoch_loss:.4f}, Train Acc: {train_acc:.2f}%, Test Acc: {test_acc:.2f}%')

Epoch 1/8:
Loss: 1.4792, Train Acc: 55.84%, Test Acc: 52.42%
Epoch 2/8:
Loss: 1.0562, Train Acc: 64.98%, Test Acc: 59.57%
Epoch 3/8:
Loss: 0.8278, Train Acc: 72.09%, Test Acc: 63.74%
Epoch 4/8:
Loss: 0.6862, Train Acc: 80.56%, Test Acc: 69.62%
Epoch 5/8:
Loss: 0.5706, Train Acc: 80.19%, Test Acc: 68.13%
Epoch 6/8:
Loss: 0.2909, Train Acc: 96.80%, Test Acc: 82.87%
Epoch 7/8:
Loss: 0.1675, Train Acc: 98.18%, Test Acc: 83.47%
Epoch 8/8:
Loss: 0.1098, Train Acc: 99.09%, Test Acc: 83.92%


проверим точность каждого класса


In [57]:
def class_accuracy(model, dataloader, class_names, device):
    model.eval()

    class_correct = [0] * len(class_names) #список для кол-ва верных предсказаний
    class_total = [0] * len(class_names) #список для общего кол-ва примеров

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            #рассматриваем результаты по каждому изображению
            for i in range(len(y)):
                label = y[i].item()
                class_total[label] += 1
                if predicted[i].item() == label:
                    class_correct[label] += 1

    total_correct = 0
    total_samples = 0

    for i in range(len(class_names)):
        if class_total[i] > 0:
            accuracy = 100 * class_correct[i] / class_total[i]
            total_correct += class_correct[i]
            total_samples += class_total[i]
            print(f"{class_names[i]}: {accuracy:6.2f}")

    overall_accuracy = 100 * total_correct / total_samples if total_samples > 0 else 0
    print(f"{'Точночть по всем классам'}: {overall_accuracy:6.2f}")

    return overall_accuracy

print("Точночть на обучающих:")
train_overall_acc = class_accuracy(model, train_dataloader, class_names, device)

print("\nТочность на тестовых:")
test_overall_acc = class_accuracy(model, test_dataloader, class_names, device)

Точночть на обучающих:
Agaricus:  98.23
Amanita:  98.17
Boletus:  99.77
Cortinarius:  98.95
Entoloma:  98.97
Hygrocybe:  97.23
Lactarius:  99.76
Russula:  99.02
Suillus:  99.20
Точночть по всем классам:  99.09

Точность на тестовых:
Agaricus:  72.86
Amanita:  84.00
Boletus:  92.56
Cortinarius:  76.65
Entoloma:  79.45
Hygrocybe:  87.30
Lactarius:  88.82
Russula:  81.30
Suillus:  72.58
Точночть по всем классам:  83.92
